# 🛰️ SUTRAM — Train on Kaggle GPU

Attach the **sutram-bundle** dataset as input, set **Accelerator = GPU T4×2** and **Internet = On**, then **Run All**. Trains Stage 1 (super-resolution) + Stage 2 (colour) on the GPU in ~10–15 min and stages the checkpoints under `/kaggle/working/sutram_trained/` for download.

In [ ]:
# ==========================================================================
# SUTRAM — Kaggle GPU training notebook
# ==========================================================================
# Paste this into a Kaggle notebook (GPU accelerator: T4 x2 or P100).
# Attach the uploaded dataset "sutram-bundle" (code + input scenes) as input.
#
# It regenerates patches from the raw scenes, then trains Stage 1 (thermal
# super-resolution) and Stage 2 (colour) on the GPU — minutes, not hours —
# and writes the trained checkpoints to /kaggle/working for download.
#
# The repo already auto-detects CUDA (training/trainer.py `_resolve_device`),
# so nothing model-side needs changing.
# ==========================================================================

import os, sys, glob, shutil, subprocess, time


In [ ]:
# --- 1. Locate the attached bundle ----------------------------------------
CANDIDATES = glob.glob("/kaggle/input/*/")
BUNDLE = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, "cli.py")):
        BUNDLE = c; break
    inner = glob.glob(os.path.join(c, "*", "cli.py"))
    if inner:
        BUNDLE = os.path.dirname(inner[0]) + "/"; break
assert BUNDLE, f"Could not find the code bundle under /kaggle/input. Saw: {CANDIDATES}"
print("bundle:", BUNDLE)


In [ ]:
# --- 2. Copy to a writable working dir ------------------------------------
ROOT = "/kaggle/working/sutram"
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)
shutil.copytree(BUNDLE, ROOT)
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("working root:", ROOT, "| scenes:", len(glob.glob("input/LC*/")))


In [ ]:
# --- 3. Dependencies (Kaggle has torch; add geo/image libs) ---------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rasterio", "tifffile", "opencv-python-headless",
                "omegaconf", "hydra-core", "scipy"], check=False)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
# --- 4. Force device=cuda in the base config ------------------------------
import re
bc = "configs/base_config.yaml"
s = open(bc).read()
s = re.sub(r'device:\s*"?\w+"?', 'device: "cuda"', s)
open(bc, "w").write(s)

# On a real GPU we can push batch size + epochs; edit configs/training.yaml.
tc = "configs/training.yaml"
t = open(tc).read()
t = t.replace("batch_size: 16", "batch_size: 32")   # GPU has the memory
open(tc, "w").write(t)

def run(cmd):
    print("\n>>>", cmd); t0 = time.time()
    r = subprocess.run(cmd, shell=True)
    print(f"<<< {cmd}  ({time.time()-t0:.0f}s, rc={r.returncode})")
    assert r.returncode == 0, cmd


In [ ]:
# --- 5. Regenerate patches from the raw scenes ----------------------------
run(f"{sys.executable} data_pipeline/prepare_dataset.py --force")
run(f"{sys.executable} scripts/prepare_sr_scenes.py")   # SR pairs from new-data thermal (optional)


In [ ]:
# --- 6. Train both stages on the GPU --------------------------------------
run(f"{sys.executable} cli.py train-stage1 --force")
run(f"{sys.executable} cli.py train-stage2 --force")
run(f"{sys.executable} cli.py evaluate")
run(f"{sys.executable} scripts/prepare_release_checkpoints.py")


In [ ]:
# --- 7. Stage the results for download -------------------------------------
OUT = "/kaggle/working/sutram_trained"
os.makedirs(OUT, exist_ok=True)
for f in glob.glob("checkpoints/*.pth") + glob.glob("experiments/*/checkpoints/*.pth"):
    shutil.copy(f, OUT)
if os.path.exists("experiments/sutram_baseline/metrics.json"):
    shutil.copy("experiments/sutram_baseline/metrics.json", OUT)
print("\nDONE. Download these from /kaggle/working/sutram_trained :")
for f in sorted(glob.glob(OUT + "/*")):
    print("  ", os.path.basename(f), f"{os.path.getsize(f)/1e6:.1f} MB")
